# 1. Data exploration

Paris PM2.5 daily means (OpenAQ). Owner: Prem — Phase 1.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))
import os
os.chdir(project_root)

from src.eda import adf_table, count_missing, impute_linear, rolling_mean_std, suggest_sarima_orders
from src.load_data import load_daily_series
from src.plots import plot_acf_pacf, plot_rolling, plot_series, save_current_figure, save_figure

DATA_PATH = project_root / "data/processed/pm25_paris_daily.csv"
(project_root / "reports/figures").mkdir(parents=True, exist_ok=True)

raw = load_daily_series(DATA_PATH)
raw.head()

date
2024-01-01    17.000000
2024-01-02    19.355198
2024-01-03    19.943599
2024-01-04    18.328985
2024-01-05    15.733609
Freq: D, Name: pm25, dtype: float64

## Full series, zoomed month, and rolling mean/std

In [2]:
ax = plot_series(raw.dropna(), title="Paris PM2.5 — full series")
save_figure(ax, "01_full_series.png")
plt.show()

/var/folders/8n/mwqyfv8j437btzxfh5c852n00000gn/T/ipykernel_23208/2040132322.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
clean = raw.dropna()
zoom_days = min(30, max(7, len(clean) - 1))
zoom_end = clean.index.max()
zoom_start = zoom_end - pd.Timedelta(days=zoom_days)
zoom = raw.loc[zoom_start:zoom_end].dropna()
ax = plot_series(zoom, title=f"Paris PM2.5 — zoomed ({zoom_start.date()} to {zoom_end.date()})")
save_figure(ax, "01_zoomed_month.png")
plt.show()

/var/folders/8n/mwqyfv8j437btzxfh5c852n00000gn/T/ipykernel_23208/3844155040.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
roll_mean, roll_std = rolling_mean_std(raw.dropna(), window=7)
ax = plot_rolling(raw.dropna(), roll_mean, roll_std, title="7-day rolling mean and std")
save_figure(ax, "01_rolling_mean_std.png")
plt.show()

/var/folders/8n/mwqyfv8j437btzxfh5c852n00000gn/T/ipykernel_23208/3275540482.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Missing timestamps and imputation

**Choice:** linear interpolation on the daily calendar (`impute_linear`), with edge `ffill`/`bfill` only if the first/last days are missing.

**Why not forward-fill:** ffill creates flat segments that artificially inflate ACF at lag 1. Short sensor outages in PM2.5 are better filled by interpolating between neighbouring days.

In [5]:
missing = count_missing(raw)
print(
    f"expected_days={missing.expected_days}\n"
    f"observed_days={missing.observed_days}\n"
    f"missing_timestamps={missing.missing_timestamps}\n"
    f"missing_values_in_file={missing.missing_values}"
)
if len(missing.missing_dates):
    print("first missing dates:", list(missing.missing_dates[:10]))

series = impute_linear(raw)
assert series.isna().sum() == 0
series.describe()

expected_days=210
observed_days=209
missing_timestamps=1
missing_values_in_file=1
first missing dates: [Timestamp('2024-02-20 00:00:00')]


count    210.000000
mean      15.797568
std        2.310303
min       11.815351
25%       13.830899
50%       15.777852
75%       17.729603
max       20.009402
Name: pm25, dtype: float64

## ADF stationarity tests

In [6]:
adf = adf_table(series)
display(adf)
raw_stationary = bool(adf.loc[adf["series"] == "raw", "stationary_5pct"].iloc[0])
print(
    "Raw series is",
    "stationary at 5%. Use d=0" if raw_stationary else "NOT stationary at 5%. Prefer d=1",
    "for SARIMA.",
)

/Users/prem/tsa-group5-air-quality/src/eda.py:56: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, pvalue, used_lag, nobs, *_ = adfuller(clean, autolag="AIC")
/Users/prem/tsa-group5-air-quality/src/eda.py:56: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, pvalue, used_lag, nobs, *_ = adfuller(clean, autolag="AIC")


,series,adf_stat,pvalue,used_lag,nobs,stationary_5pct
0,raw,-2.154243,0.223269,15,194,False
1,diff_1,-0.581568,0.875093,15,193,False


Raw series is NOT stationary at 5%. Prefer d=1 for SARIMA.


## ACF/PACF and suggested orders

Inspect lag-1 behaviour for `(p, q)` and lags 7 / 14 for weekly seasonal `(P, Q, s=7)`.

In [7]:
plot_acf_pacf(series, lags=40)
save_current_figure("01_acf_pacf.png")
plt.show()

prefer_diff = not raw_stationary
orders = suggest_sarima_orders(seasonal_period=7, prefer_diff=prefer_diff)
print("Suggested order:", orders["order"])
print("Suggested seasonal_order:", orders["seasonal_order"])
print("Hand these to Yahya (also in docs/sarima_orders_for_yahya.md).")

Suggested order: (1, 1, 1)
Suggested seasonal_order: (1, 1, 1, 7)
Hand these to Yahya (also in docs/sarima_orders_for_yahya.md).


/var/folders/8n/mwqyfv8j437btzxfh5c852n00000gn/T/ipykernel_23208/2444714836.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
